In [88]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

date_etudie = '2017-03-20'

In [89]:
try:
    file_path = "TOTF_book_03_04_2017.csv"
    data_book = pd.read_csv(file_path)
except FileNotFoundError:
    file_path = "../TOTF_book_03_04_2017.csv"
    data_book = pd.read_csv(file_path)

In [90]:
try:
    file_path2 = "TOTF_trade_2014_2017.csv"
    data_trade_init = pd.read_csv(file_path2)
except FileNotFoundError:
    file_path2 = "../TOTF_trade_2014_2017.csv"
    data_trade_init = pd.read_csv(file_path2)


In [91]:
data_trade = data_trade_init.drop(columns=['source_file'])
data_trade = data_trade[(data_trade['date'] == date_etudie)]

In [92]:
columns_to_drop = ['bid_2',           'ask_2',  
                   'bid_3', 'bidQ_3', 'ask_3', 'askQ_3', 
                   'bid_4', 'bidQ_4', 'ask_4', 'askQ_4', 
                   'bid_5', 'bidQ_5', 'ask_5', 'askQ_5', 
                   'source_file']
data_book_1 = data_book.drop(columns=columns_to_drop)

data_book_2 = data_book_1[~((data_book_1['askQ_1'] == data_book['askQ_1'].shift()) & 
                           (data_book_1['bidQ_1'] == data_book['bidQ_1'].shift()) &
                           (data_book_1['askQ_2'] == data_book['askQ_2'].shift()) & 
                           (data_book_1['bidQ_2'] == data_book['bidQ_2'].shift()))]

data_book_2 = data_book_2[(data_book_2['date'] == date_etudie) ]
data_book_2.reset_index(drop=True, inplace=True)

In [93]:
# Create a new DataFrame with the specified columns and the same number of rows as data_book_2
new_table = pd.DataFrame({
    'date': data_book_2['date'],
    'time': data_book_2['time'],
    'V_lo_b': np.zeros(len(data_book_2), dtype='float64'),
    'V_c_b': np.zeros(len(data_book_2), dtype='float64'),
    'V_ex_b': np.zeros(len(data_book_2), dtype='float64'),
    'V_lo_a': np.zeros(len(data_book_2), dtype='float64'),
    'V_c_a': np.zeros(len(data_book_2), dtype='float64'),
    'V_ex_a': np.zeros(len(data_book_2), dtype='float64'),
    'mid_price': np.zeros(len(data_book_2), dtype='float64'),
    'bid_1': data_book_2['bid_1'],
    'ask_1': data_book_2['ask_1'],
})


In [94]:
bid_1_initial = data_book_2.loc[0, 'bid_1']
bidQ_1_initial = data_book_2.loc[0, 'bidQ_1']
ask_1_initial = data_book_2.loc[0, 'ask_1']
askQ_1_initial = data_book_2.loc[0, 'askQ_1']
askQ_2_initial = data_book_2.loc[0, 'askQ_2']
bidQ_2_initial = data_book_2.loc[0, 'bidQ_2']



for i in range(1,len(data_book_2)):
    askQ_2_new = data_book_2.iloc[i]['askQ_2']
    bidQ_2_new = data_book_2.iloc[i]['bidQ_2']
    bid_1_new = data_book_2.iloc[i][ 'bid_1']
    bidQ_1_new = data_book_2.iloc[i]['bidQ_1']
    ask_1_new = data_book_2.iloc[i]['ask_1']
    askQ_1_new = data_book_2.iloc[i][ 'askQ_1']
    date_new = data_book_2.iloc[i]['date']
    heure_new = data_book_2.iloc[i]['time']
    
    
    if bid_1_new < bid_1_initial:
        new_table.loc[i, 'V_c_b'] = bidQ_1_initial + np.max((bidQ_2_initial - bidQ_1_new),0)
    elif bid_1_new > bid_1_initial:
        new_table.loc[i, 'V_lo_b'] = bidQ_1_new + np.max((bidQ_2_new - bidQ_1_initial),0)
    else: 
        if bidQ_1_new + bidQ_2_new > bidQ_1_initial + bidQ_2_initial:
            new_table.loc[i, 'V_lo_b'] = bidQ_1_new - bidQ_1_initial + bidQ_2_new - bidQ_2_initial
        if bidQ_1_new + bidQ_2_new < bidQ_1_initial + bidQ_2_initial:
            new_table.loc[i, 'V_c_b'] = bidQ_1_initial - bidQ_1_new + bidQ_2_initial - bidQ_2_new
                
    if ask_1_new  > ask_1_initial :
        new_table.loc[i,'V_c_a'] = askQ_1_initial + np.max((askQ_2_initial - askQ_1_new),0)
    elif ask_1_new < ask_1_initial :
        new_table.loc[i,'V_lo_a'] = askQ_1_new + np.max(askQ_2_new - askQ_1_initial,0)          
    else :
        if askQ_1_new + askQ_2_new > askQ_1_initial + askQ_2_initial:
            new_table.loc[i,'V_lo_a'] = askQ_1_new - askQ_1_initial + askQ_2_new - askQ_2_initial
        if askQ_1_new + askQ_2_new < askQ_1_initial + askQ_2_initial:
            new_table.loc[i,'V_c_a'] = askQ_1_initial - askQ_1_new + askQ_2_initial - askQ_2_new
        
    new_table.loc[i, 'mid_price'] = (bid_1_new + ask_1_new) / 2
    bid_1_initial = bid_1_new
    bidQ_1_initial = bidQ_1_new
    ask_1_initial = ask_1_new
    askQ_1_initial = askQ_1_new
    bidQ_2_initial = bidQ_2_new
    askQ_2_initial = askQ_2_new
    


In [95]:
somme_1_bid = 0
somme_1_ask = 0
somme_2_bid = 0
somme_2_ask = 0
compteur = 0

for i in range(0,len(data_trade)):
    trade_date = data_trade.iloc[i]['date']
    trade_time = data_trade.iloc[i]['time']
    trade_price = float(data_trade.iloc[i]['trade.price'])
    
    trouve = False
    trade_volume = data_trade.iloc[i]['trade.volume']
    data_table_bid = new_table[(new_table['date'] == trade_date) & 
                               (new_table['time'] == trade_time) & 
                               (np.abs(new_table['bid_1'] - trade_price) <= new_table['mid_price'])]
    
    data_table_ask = new_table[(new_table['date'] == trade_date) & 
                               (new_table['time'] == trade_time) & 
                               (np.abs(new_table['ask_1'] - trade_price) <= new_table['mid_price'])]
   
    if not data_table_ask.empty and not trouve:
        data_table_ask_exact = data_table_ask[data_table_ask['V_c_a'] == trade_volume ]
        if not data_table_ask_exact.empty:
            new_table.loc[data_table_ask_exact.index[0], 'V_ex_a'] = trade_volume
            new_table.loc[data_table_ask_exact.index[0], 'V_c_a'] = 0
            somme_1_ask += 1
            trouve = True
            
        else :
            data_table_bid_not_exact = data_table_ask[(data_table_ask['V_c_a'] > trade_volume)]
            if not data_table_bid_not_exact.empty:
                new_table.loc[data_table_bid_not_exact.index[0], 'V_ex_a'] = trade_volume
                new_table.loc[data_table_bid_not_exact.index[0], 'V_c_a'] -= trade_volume
                somme_2_ask += 1
                trouve = True
    if not data_table_bid.empty and not trouve:
        data_table_bid_exact = data_table_bid[data_table_bid['V_c_b'] == trade_volume]
        if not data_table_bid_exact.empty:
            new_table.loc[data_table_bid_exact.index[0], 'V_ex_b'] = trade_volume
            new_table.loc[data_table_bid_exact.index[0], 'V_c_b'] = 0
            somme_1_bid += 1
            trouve = True
        else : 
            data_table_bid_not_exact = data_table_bid[(data_table_bid['V_c_b'] > trade_volume)]
            if not data_table_bid_not_exact.empty:
                new_table.loc[data_table_bid_not_exact.index[0], 'V_ex_b'] = trade_volume
                new_table.loc[data_table_bid_not_exact.index[0], 'V_c_b'] -= trade_volume
                somme_2_bid += 1
                trouve = True
    if not trouve:
        compteur +=1
    

In [96]:
new_table.head(50)

,date,time,V_lo_b,V_c_b,V_ex_b,V_lo_a,V_c_a,V_ex_a,mid_price,bid_1,ask_1
0,2017-03-20,09:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,46.320,46.325
1,2017-03-20,09:30:00,0.0,0.0,0.0,0.0,200.0,0.0,46.3225,46.320,46.325
2,2017-03-20,09:30:00,222.0,0.0,0.0,0.0,0.0,0.0,46.3225,46.320,46.325
3,2017-03-20,09:30:00,195.0,0.0,0.0,0.0,0.0,0.0,46.3225,46.320,46.325
4,2017-03-20,09:30:01,160.0,0.0,0.0,0.0,0.0,0.0,46.3225,46.320,46.325
5,2017-03-20,09:30:03,200.0,0.0,0.0,0.0,0.0,0.0,46.3225,46.320,46.325
6,2017-03-20,09:30:03,200.0,0.0,0.0,0.0,0.0,608.0,46.3225,46.320,46.325
7,2017-03-20,09:30:03,0.0,0.0,0.0,0.0,157.0,31.0,46.3225,46.320,46.325
8,2017-03-20,09:30:03,0.0,0.0,0.0,0.0,410.0,0.0,46.3250,46.320,46.330
9,2017-03-20,09:30:03,0.0,0.0,0.0,0.0,200.0,0.0,46.3250,46.320,46.330


In [97]:

new_table['bid_1'] = data_book_2['bid_1']
new_table['ask_1'] = data_book_2['ask_1']
new_table.to_csv(f"new_table_{date_etudie}_compexe.csv")

In [98]:
data_trade.head()

,date,time,trade.price,trade.volume
5213740,2017-03-20,09:30:03,46.325,608.0
5213741,2017-03-20,09:30:03,46.325,598.0
5213742,2017-03-20,09:30:03,46.330,31.0
5213743,2017-03-20,09:30:05,46.330,160.0
5213744,2017-03-20,09:30:08,46.325,242.0


In [99]:
filtered_table_2 = new_table[(new_table['V_ex_b'] != 0) | (new_table['V_ex_a'] != 0)]
print(len(filtered_table_2))
print(len(data_trade))
print(f'We detected {(len(filtered_table_2)/len(data_trade))*100}%.2f trades in the trade file.')

print(somme_1_bid, somme_2_bid, somme_1_ask, somme_2_ask,compteur)

3920
5603
We detected 69.96252007852935%.2f trades in the trade file.
310 488 1317 1919 1569


In [100]:

# Construire une table avec les lignes où un des volumes est négatif
negative_volume_table = new_table[(new_table['V_lo_b'] < 0) | 
                                  (new_table['V_c_b'] < 0) | 
                                  (new_table['V_ex_b'] < 0) | 
                                  (new_table['V_lo_a'] < 0) | 
                                  (new_table['V_c_a'] < 0) | 
                                  (new_table['V_ex_a'] < 0)]
negative_volume_table.head()

,date,time,V_lo_b,V_c_b,V_ex_b,V_lo_a,V_c_a,V_ex_a,mid_price,bid_1,ask_1
